# Demo: Dataset Middleware

An agent builds an RDF graph through middleware-provided tools, and we inspect the run with live tracing.

---

- **What is `DatasetMiddleware`?** A set of tools, prompts, and supporting infrastructure for LangChain agents.
- **What problem does it solve for a Research Agent?** Frontier models are improving their logical reasoning and mathematical skills all the time.
   - A structured memory is a formal representation of facts, which can **reduce cognitive burden**; e.g., fewer context tokens and fewer agent reasoning cycles.
   - Agents are encouraged to express facts in an _atemporal_ description logic; this avoids the problem of **stale facts accumulating in agent history**. Instead, a bounded description about some facts would automatically describe temporal context and any other necessary metadata.
- **What will this notebook demonstrate?** How to get an agent converting unstructured text into structured knowledge as quickly as possible. Also, how to trace the agent's activity while it does so.

## 1. Setup

We've got a few things to do before the demo gets started.

First, we have to filter a langchain-internal warning (unrelated to our data model, it's just annoying).

In [1]:
import warnings

warnings.filterwarnings(
    "ignore",
    message="Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.",
    category=UserWarning,
)

### 1.1. Bring your own Model

This notebook uses [LangChain's OpenAI integration](https://docs.langchain.com/oss/python/integrations/chat/openai) for [a self-hosted model provider](https://docs.langchain.com/langsmith/playground-model-providers#openai-compatible-endpoint), but you can use any of [LangChain's supported providers](https://docs.langchain.com/oss/python/integrations/providers/all_providers)
What's important is that Deepagents requires a chat model capable of tool invocations.

_NOTE: The only reason for `streaming=False` is that the Ministral model that I'm using starts to embed tool invocations in its normal chat output, causing us to lose tool calls.
Your mileage may vary depending on the model you select._

In [2]:
import os

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model=os.getenv("CHAT_MODEL"),
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_BASE_URL"),
    temperature=1,
    streaming=False,  # NOTE: True -> problems for mistral's tool call responses
)

## 2. The Demonstration

### 2.1. Prompt Setup

- You'll want to set up your prompt for your particular use case.
   - Here, we give the agent the persona of a specialist; the agent should think that it has an analytical mind, which can direct its attention toward skepticism and challenging its first impressions.
- `DatasetMiddleware` automatically educates the agent about their knowledge representation capabilities through `LangChain`'s middleware pattern. Specifically, [we extend the prompt](../rdflib-reasoning-middleware/src/rdflibr/middleware/dataset_middleware.py) to tell the agent about tools and how to use them.
- Furthermore, we [rigorously annotate our Pydantic types](../rdflib-reasoning-middleware/src/rdflibr/middleware/dataset_model.py), giving agents specification-backed references as well as clean examples.
   - An agent with web search or URL retrieval capabilities can dig as deep as they need to in order to understand what they are working on. This is especially helpful if an agent seems to struggle with the meaning of semantic graphs.


Something important to note here is that the prompt is clearly **trying** to ground the agent in RDF modeling.
The agent is given an explicit set of constraints on what kind of vocabulary terms it should use, but _it will not be able to satisfy those constraints_.
Crucially, the agent has no affordances to expand its context dynamically with background information.
Even if it decides to check its work for compliance, it has no way to conclude that it _is_ in violation of the prompt.

In [3]:
from typing import Final

SYSTEM_PROMPT: Final[str] = """
You are a research assistant specialized in ontological modeling and the semantic web.
You are given a question to answer or a task to complete.

You have access to tools that allow you to curate a RDF knowledge base, taking advantage of your specialization.
When appropriate, you SHOULD present your answer, reasoning, or supporting evidence in `text/turtle`.

When transforming unstructured text into RDF, you MUST attempt to use controlled RDF vocabularies.
You MUST ensure that your asserted facts are grounded in the provided content; they MUST NOT be assumptions or inferences, unless you have been requested to do so as part of your task.
You MUST NOT assert facts that you are uncertain about given the provided content.
You SHOULD assert all facts that are claimed by the provided content.

You MUST use `add_triples` to add triples to your knowledge base.
You MUST use `serialize_dataset` to serialize your knowledge base before formulating your response.
Your output MUST include the serialized graph as a `text/turtle` block.
"""

### 2.2. Agent Task Definition

Here we finally present the agent with a problem: Extract an ontology from this data.

- We are deliberately vague, simply asking if we can formalize the data
   - The given persona tells the agent to focus on the semantic modeling problems that are a throughline of the text.
- We add extraneous information, which we shouldn't see in the output.
- Because this demonstration doesn't use `RDFVocabularyMiddleware`, the agent doesn't have the ability to find established vocabularies. Instead it will just make up terms. After all, we're asking it to assert **ALL** facts that are claimed by the content.
   - It's not enough to ask an agent to be grounded; you have to afford it references that it can use to ground itself.

In [4]:
TASK: Final[str] = """
Please assist me in representing the subject matter of the following text as an RDF graph.
Please use <urn:ex:> as the base for any IRIs that you need to mint as part
of your response. Be sure to label any new terms or properties that you mint
so that they are human readable.

```text
John is a person. Modern people are classified as _homo sapiens_. Apparently,
homo sapiens falls under the subtribe of Hominina. Every Hominidae is a
Haplorhini, and now the names don't even sound like they're related. Then
we get to primates, which can be controversial for some people, for some reason.
Nobody argues with the idea that primates are mammals, yet some people take
umbrage with the idea that _homo sapiens_ is an animal. Oh, Hominidae contains
Hominina, too. Emotions and theological views aside, can we formalize this?
```
"""

#### 2.2.1 Desired Output

The agent doesn't have much (any) background knowledge, so we're not going to penalize it for missing best-practices:

1. Concepts that this document introduces sould be documented:
    1.1. They SHOULD have a [`rdfs:label`](https://www.w3.org/TR/rdf-schema/#ch_label)
    1.2. They SHOULD have a [`skos:definition`](https://www.w3.org/TR/skos-reference/#notes)
    1.2. They SHOULD have a description in the form of [`dcterms:description`](https://www.dublincore.org/specifications/dublin-core/dcmi-terms/#description), or at least a [`rdfs:comment`](https://www.w3.org/TR/rdf-schema/#ch_comment)

In [5]:
from rdflib import Graph

expected = Graph()
expected.parse(
    data="""
@prefix ex: <urn:ex:> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .

ex:John a ex:Person .

# NOTE: We wouldn't expect owl:equivalentClass unless the agent knows OWL
ex:Person rdfs:subClassOf ex:HomoSapiens ;
          owl:equivalentClass ex:HomoSapiens .

ex:HomoSapiens rdfs:subClassOf ex:Hominina .

ex:Hominina rdfs:subClassOf ex:Hominidae .

ex:Hominidae rdfs:subClassOf ex:Haplorhini .

ex:Haplorhini rdfs:subClassOf ex:Primates .

ex:Primates rdfs:subClassOf ex:Mammals .

ex:Mammals rdfs:subClassOf ex:Animal .

"""
)

print(f"The expected graph contains {len(expected)} triples")

The expected graph contains 9 triples


### 2.3. Tracing the Agent

Now, a decent model will handle this problem just fine without the dataset middleware.
Without prompting, it may decide that there's no reason to use the tools and to just craft RDF as `text/turtle` directly.
This is risky, as a single syntax error means that the agent's output is useless for an integrated system.

The goal is to create _structured_ output.
The agent's population of a dataset serves two purposes:

1. The agent is able to reflect on the state of the dataset rather than its memory of the dataset.
2. The agent is able to output structured data without forcing it to interpret a complex schema.

#### 2.3.1. Traces for Humans

We want to run an agent interactively within a notebook and we don't want to see a thousand lines of JSON. We want to take advantage of the environment's rendering capabilities.
The `LiveNotebookTrace` is a context manager supplying a wrapper for an agent.
While attached, it will accumulate and report events from agent callbacks.
If the agent completes successfully or exceptionally, the trace detaches from the agent and the notebook, keeping the environment stable.

In [6]:
from deepagents import create_deep_agent
from rdflibr.middleware.dataset_middleware import DatasetMiddleware
from rdflibr.middleware.ministral_middleware import MinistralPromptSuffixMiddleware
from rdflibr.middleware.tracing_notebook import LiveNotebookTrace

dataset_middleware = DatasetMiddleware()

with LiveNotebookTrace(heading="Dataset Trace") as trace:
    agent = trace.attach(
        create_deep_agent(
            model=llm,
            system_prompt=SYSTEM_PROMPT,
            middleware=[
                dataset_middleware,
                MinistralPromptSuffixMiddleware(),
            ],
        )
    )

    result = agent.invoke({"messages": [{"role": "user", "content": TASK}]})

# Dataset Trace

## Turn 1

### Tool: add_triples

#### Arguments

```json
{
  "triples": [
    {
      "object": "<urn:ex:Person>",
      "predicate": "<http://www.w3.org/1999/02/22-rdf-syntax-ns#type>",
      "subject": "<urn:ex:John>"
    },
    {
      "object": "\"Person\"",
      "predicate": "<http://www.w3.org/2000/01/rdf-schema#label>",
      "subject": "<urn:ex:Person>"
    },
    {
      "object": "\"A biological classification for humans.\"",
      "predicate": "<http://www.w3.org/2000/01/rdf-schema#comment>",
      "subject": "<urn:ex:Person>"
    }
  ]
}
```

#### Result

```json
updated=3 message='Triples added to the default graph.'
```

## Turn 2

### Tool: add_triples

#### Arguments

```json
{
  "triples": [
    {
      "object": "\"Homo Sapiens\"",
      "predicate": "<http://www.w3.org/2000/01/rdf-schema#label>",
      "subject": "<urn:ex:HomoSapiens>"
    },
    {
      "object": "\"A modern human species classified under the subtribe Hominina.\"",
      "predicate": "<http://www.w3.org/2000/01/rdf-schema#comment>",
      "subject": "<urn:ex:HomoSapiens>"
    },
    {
      "object": "<urn:ex:HomoSapiens>",
      "predicate": "<http://www.w3.org/2000/01/rdf-schema#subClassOf>",
      "subject": "<urn:ex:Person>"
    }
  ]
}
```

#### Result

```json
updated=3 message='Triples added to the default graph.'
```

## Turn 3

### Tool: add_triples

#### Arguments

```json
{
  "triples": [
    {
      "object": "\"Hominina\"",
      "predicate": "<http://www.w3.org/2000/01/rdf-schema#label>",
      "subject": "<urn:ex:Hominina>"
    },
    {
      "object": "\"A subtribe under which Homo Sapiens is classified.\"",
      "predicate": "<http://www.w3.org/2000/01/rdf-schema#comment>",
      "subject": "<urn:ex:Hominina>"
    },
    {
      "object": "<urn:ex:Hominina>",
      "predicate": "<http://www.w3.org/2000/01/rdf-schema#subClassOf>",
      "subject": "<urn:ex:HomoSapiens>"
    }
  ]
}
```

#### Result

```json
updated=3 message='Triples added to the default graph.'
```

## Turn 4

### Tool: add_triples

#### Arguments

```json
{
  "triples": [
    {
      "object": "\"Hominidae\"",
      "predicate": "<http://www.w3.org/2000/01/rdf-schema#label>",
      "subject": "<urn:ex:Hominidae>"
    },
    {
      "object": "\"A family of great apes that includes Hominina.\"",
      "predicate": "<http://www.w3.org/2000/01/rdf-schema#comment>",
      "subject": "<urn:ex:Hominidae>"
    },
    {
      "object": "<urn:ex:Haplorhini>",
      "predicate": "<http://www.w3.org/2000/01/rdf-schema#subClassOf>",
      "subject": "<urn:ex:Hominidae>"
    }
  ]
}
```

#### Result

```json
updated=3 message='Triples added to the default graph.'
```

## Turn 5

### Tool: add_triples

#### Arguments

```json
{
  "triples": [
    {
      "object": "\"Haplorhini\"",
      "predicate": "<http://www.w3.org/2000/01/rdf-schema#label>",
      "subject": "<urn:ex:Haplorhini>"
    },
    {
      "object": "\"A suborder of primates that includes Hominidae.\"",
      "predicate": "<http://www.w3.org/2000/01/rdf-schema#comment>",
      "subject": "<urn:ex:Haplorhini>"
    },
    {
      "object": "<urn:ex:Primates>",
      "predicate": "<http://www.w3.org/2000/01/rdf-schema#subClassOf>",
      "subject": "<urn:ex:Hominidae>"
    }
  ]
}
```

#### Result

```json
updated=3 message='Triples added to the default graph.'
```

## Turn 6

### Tool: add_triples

#### Arguments

```json
{
  "triples": [
    {
      "object": "\"Primates\"",
      "predicate": "<http://www.w3.org/2000/01/rdf-schema#label>",
      "subject": "<urn:ex:Primates>"
    },
    {
      "object": "\"An order of mammals that includes Hominidae.\"",
      "predicate": "<http://www.w3.org/2000/01/rdf-schema#comment>",
      "subject": "<urn:ex:Primates>"
    },
    {
      "object": "<urn:ex:Mammals>",
      "predicate": "<http://www.w3.org/2000/01/rdf-schema#subClassOf>",
      "subject": "<urn:ex:Primates>"
    }
  ]
}
```

#### Result

```json
updated=3 message='Triples added to the default graph.'
```

## Turn 7

### Tool: add_triples

#### Arguments

```json
{
  "triples": [
    {
      "object": "\"Mammals\"",
      "predicate": "<http://www.w3.org/2000/01/rdf-schema#label>",
      "subject": "<urn:ex:Mammals>"
    },
    {
      "object": "\"A class of warm-blooded animals that includes primates.\"",
      "predicate": "<http://www.w3.org/2000/01/rdf-schema#comment>",
      "subject": "<urn:ex:Mammals>"
    }
  ]
}
```

#### Result

```json
updated=2 message='Triples added to the default graph.'
```

## Turn 8

### Tool: add_triples

#### Arguments

```json
{
  "triples": [
    {
      "object": "<urn:ex:Hominina>",
      "predicate": "<http://www.w3.org/2000/01/rdf-schema#subClassOf>",
      "subject": "<urn:ex:Hominidae>"
    }
  ]
}
```

#### Result

```json
updated=1 message='Triples added to the default graph.'
```

## Turn 9

### Tool: serialize_dataset

#### Arguments

```json
{
  "format": "turtle"
}
```

#### Result

```json
format='turtle' content='@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .\n\n<urn:ex:Hominidae> rdfs:label "Hominidae" ;\n    rdfs:comment "A family of great apes that includes Hominina." ;\n    rdfs:subClassOf <urn:ex:Haplorhini>,\n        <urn:ex:Hominina>,\n        <urn:ex:Primates> .\n\n<urn:ex:John> a <urn:ex:Person> .\n\n<urn:ex:Haplorhini> rdfs:label "Haplorhini" ;\n    rdfs:comment "A suborder of primates that includes Hominidae." .\n\n<urn:ex:HomoSapiens> rdfs:label "Homo Sapiens" ;\n    rdfs:comment "A modern human species classified under the subtribe Hominina." ;\n    rdfs:subClassOf <urn:ex:Hominina> .\n\n<urn:ex:Mammals> rdfs:label "Mammals" ;\n    rdfs:comment "A class of warm-blooded animals that includes primates." .\n\n<urn:ex:Person> rdfs:label "Person" ;\n    rdfs:comment "A biological classification for humans." ;\n    rdfs:subClassOf <urn:ex:HomoSapiens> .\n\n<urn:ex:Primates> rdfs:label "Primates" ;\n    rdfs:comment "An order of mammals that includes Hominidae." ;\n    rdfs:subClassOf <urn:ex:Mammals> .\n\n<urn:ex:Hominina> rdfs:label "Hominina" ;\n    rdfs:comment "A subtribe under which Homo Sapiens is classified." .\n\n'
```

## Turn 10

### Final Response

> Here is the RDF graph representing the content provided in the text:
>
> ```text/turtle
> @prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
> @prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
> @prefix xsd: <http://www.w3.org/2001/XMLSchema#> .
> @prefix urn:ex: <urn:ex:> .
>
> <urn:ex:Hominidae> rdfs:label "Hominidae" ;
>     rdfs:comment "A family of great apes that includes Hominina." ;
>     rdfs:subClassOf <urn:ex:Haplorhini>,
>         <urn:ex:Hominina>,
>         <urn:ex:Primates> .
>
> <urn:ex:John> a <urn:ex:Person> .
>
> <urn:ex:Haplorhini> rdfs:label "Haplorhini" ;
>     rdfs:comment "A suborder of primates that includes Hominidae." .
>
> <urn:ex:HomoSapiens> rdfs:label "Homo Sapiens" ;
>     rdfs:comment "A modern human species classified under the subtribe Hominina." ;
>     rdfs:subClassOf <urn:ex:Hominina> .
>
> <urn:ex:Mammals> rdfs:label "Mammals" ;
>     rdfs:comment "A class of warm-blooded animals that includes primates." .
>
> <urn:ex:Person> rdfs:label "Person" ;
>     rdfs:comment "A biological classification for humans." ;
>     rdfs:subClassOf <urn:ex:HomoSapiens> .
>
> <urn:ex:Primates> rdfs:label "Primates" ;
>     rdfs:comment "An order of mammals that includes Hominidae." ;
>     rdfs:subClassOf <urn:ex:Mammals> .
>
> <urn:ex:Hominina> rdfs:label "Hominina" ;
>     rdfs:comment "A subtribe under which Homo Sapiens is classified." .
> ```


### 2.4. Manually Assessing Results

As we expected, the agent used almost no standard vocabularies.
The agent _did_ use `rdf:type`, but doing so only suggests that the agent is aware that RDF _exists_.
The problem implied by the initial text is establishing a taxonomy of types and finding out where John fits into it.
This is exactly the kind of modeling that RDFS supports trivially, yet the agent is lacking the technical context.

Ultimately, we demonstrate tools that allow the agent to structure and model the data,
but the agent does not demonstrate any implicit understanding of _how_ to model the data, or _what_ it should use to describe it.
A step in the right direction would be to allow the agent to explore vocabularies, and dynamically build its context.

### 2.5. Accessing the Knowledge Base

You may have noticed that we retained a reference to the middleware when creating the agent:

```python
dataset_middleware = DatasetMiddleware()
```

At any point (even during agent execution) you can take a snapshot of the current dataset.
This is a useful method to extract the structured knowledge base without asking the agent to understand RDF serialization formats.

In [7]:
from rdflib import Dataset

dataset: Dataset = dataset_middleware.session.snapshot_dataset()

print("There are", len(dataset), "triples in the dataset")
print("-" * 80)
print(dataset.serialize(format="turtle"))

There are 21 triples in the dataset
--------------------------------------------------------------------------------
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

<urn:ex:Hominidae> rdfs:label "Hominidae" ;
    rdfs:comment "A family of great apes that includes Hominina." ;
    rdfs:subClassOf <urn:ex:Haplorhini>,
        <urn:ex:Hominina>,
        <urn:ex:Primates> .

<urn:ex:John> a <urn:ex:Person> .

<urn:ex:Haplorhini> rdfs:label "Haplorhini" ;
    rdfs:comment "A suborder of primates that includes Hominidae." .

<urn:ex:HomoSapiens> rdfs:label "Homo Sapiens" ;
    rdfs:comment "A modern human species classified under the subtribe Hominina." ;
    rdfs:subClassOf <urn:ex:Hominina> .

<urn:ex:Mammals> rdfs:label "Mammals" ;
    rdfs:comment "A class of warm-blooded animals that includes primates." .

<urn:ex:Person> rdfs:label "Person" ;
    rdfs:comment "A biological classification for humans." ;
    rdfs:subClassOf <urn:ex:HomoSapiens> .

<urn:ex:Primates> rdfs:label "P

### 2.6. Metrics

We have an intersting set of problems in terms of assessing our agent's performance:

1. We are comparing sets (which constrains choices of metrics)
2. If a fact missing from our prediction set, it's _likely_ bad
3. If our prediction set contains additional facts, it's _not necessarily_ bad.

#### 2.6.1. Why Are Facts Missing?

If a fact is _missing_, then the agent probably worked around it.
It probably made up new prompt-relative RDF Vocabulary terms.

One normally needs to create at least _some_ domain-specific terms when applying semantic web technologies to novel areas, but it's a sledgehammer.
Each new term lacks shared semantics and requires consumers to learn and align with it.
Adding more application-specific terms makes it easier to express your data, but makes it less useful it is to other agents (human or otherwise).d

There's some alternative explanations for missing facts:

1. What if the agent didnt model things explicitly as classes, but as `skos:Concept`s?
Well, that would be _less_ correct, but it wouldn't be impossible to understand.

2. What if the agent uses a more relevent and published domain-specific vocabulary; one that it didn't make up?
In that respect, the agent would be _more_ correct than our labeled examples.
This isn't super likely, beacuse most agents don't have every RDF vocabulary memorized.

> **Requirement:** We need tooling for agents to find and explore new vocabularies.

#### 2.6.2. Extra Facts?

We aren't penalizing the agent for failing to adhere to best practices, but tha doesn't mean that they _won't_.
This may introduce additional documentation triples (`rdfs:label`, `skos:definition`, etc).

We may also get richer semantics.
[OWL Object Property Restrctions](https://www.w3.org/TR/2012/REC-owl2-syntax-20121211/#Object_Property_Restrictions)
could tell us about relationships that a new class is allowed to have, or certain relationships _exist_ even if they aren't
explicitly represented in the data.
[Shapes Constraint Language](https://www.w3.org/TR/shacl/) could tell us about the graph patterns that a concept is allowed to have, which tells us about graph neighborhoods that are, again, not necessarily in the data.

Agents can't do either of these things in a zero-shot scenario if they aren't aware that they are even things that they can do.

> **Requirement:** Agents needs affordances and the ability to build model patterns.

#### 2.6.3. Are Metrics Insurmountable?

The problems that I described above are those faced by agents with more sophisticated behavior.
An agent with little-to-no RDF knowledge attempting to solve this problem without any investigative tools has the deck stacked against them.
In my typical testing at this level of middleware, the agent has a significant tendency to invent vocabulary.
It's not often _wrong_, but it's **creating** a new and isolated ontology.

Given these constraints, _this_ particular model can be asssessed by 

## 3. Conclusions

> What did dataset middleware give us?

The middleware by itself simply gave the agent the ability to modify a knowledge base, and encouragement to do so when needed.
The prompt built into the middleware gives some general guidance as to how _facts_ should be represented, but it does not push the agent towards an understanding of _what the agent **should** model with those facts_.

> What comes next?

We'll allow an agent to explore standard vocabularies.
This will help to ground the agent's terminology.
Vocabularies describe how their terms should be used and what their use means.
This falls short of prescriptive guidance towards how the agent should model knowledge, but it should help the agent produce higher-quality results.